# CE541E08 — Unit 2 · Day 16 — break, continue and pass: Jump Statements for Data Quality Control
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 2 |
| **Session** | Day 16 of 45 |
| **CO** | CO2 |
| **Topics** | continue · break · pass · skip missing data · detect flood peak · sensor QC |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
github_repo  = "https://github.com/your-username/CE541E08-2026"
session      = "Day 16"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Loop Control Statements

Three statements modify the normal flow of a loop:

| Statement | Effect | When to use |
|---|---|---|
| `continue` | Skip the rest of this iteration; go to next | Skip bad records, missing values |
| `break` | Exit the loop immediately | Stop when event detected, early termination |
| `pass` | Do nothing (placeholder) | Empty block that will be filled later |

`continue` and `break` work in both `for` and `while` loops. They make loops more efficient — no need to nest deeply or set flag variables.

---
## Code Block 1 — continue: Skip Missing Sensor Readings

### What this code does

We process a CWC streamflow record that contains -999 flags for missing readings. Using `continue`, we skip each missing record and process only valid values.

### Why each step is taken

**`if Q == -999:`:**
Check for the missing-value sentinel before any computation. If found, print a skip message and `continue` — which jumps immediately to the next iteration, skipping all remaining lines in the loop body.

**Why `continue` is better than a nested `if`:**
Without `continue`, we would need to wrap all processing code in `if Q != -999:`. With `continue`, the valid-data processing code is not indented — it stays at the top level. This reduces nesting and is easier to read.

**`valid_count` and `valid_total`:**
Accumulators that only receive contributions on valid iterations (those not skipped by `continue`).

### Algorithm

```
1. For each Q in streamflow:
     if Q == -999:
       print "MISSING skipped"
       continue         ← jump to next iteration
     valid_total += Q
     valid_count += 1
     if Q > peak: peak = Q

2. After loop:
   mean = valid_total / valid_count
   print summary
```

### Expected output

```
  Record  3: MISSING (-999) — skipped
  Record  6: MISSING (-999) — skipped
  Record  9: MISSING (-999) — skipped
  Record 13: MISSING (-999) — skipped
  Record 17: MISSING (-999) — skipped
Valid: 13  Mean: 347.9  Peak: 890.2 m3/s
```

In [ ]:
streamflow = [
    234.5, 267.8, -999, 312.4, 890.2, -999, 756.4,
    543.2, -999, 345.6, 289.4, 245.1, -999, 198.7,
    212.3, 178.9, -999, 156.4,
]

valid_total = 0.0
valid_count = 0
peak_flow   = 0.0

for i, Q in enumerate(streamflow, 1):
    if Q == -999:
        print(f"  Record {i:>2}: MISSING (-999) — skipped")
        continue           # jump to next iteration — skip processing below

    # Only reached for valid (non-missing) records
    valid_total += Q
    valid_count += 1
    if Q > peak_flow:
        peak_flow = Q

mean_flow = valid_total / valid_count
print(f"Valid: {valid_count}  Mean: {mean_flow:.1f}  Peak: {peak_flow} m3/s")

### 🔁 Try this

Add a second QC check: skip values > 5000 m³/s as physically impossible spikes.

After the -999 check, add:

```python
if Q > 5000:
    print(f"  Record {i:>2}: SPIKE ({Q}) — skipped")
    continue
```

Add a spike value to the list and verify it gets skipped.

---
## Code Block 2 — break: Detect First Flood Stage Day

### What this code does

We scan a daily stage record to find the FIRST day when the river stage exceeds the flood stage threshold. Once found, we issue the alert and exit the loop — no need to scan the remaining days.

### Why each step is taken

**`if stage >= flood_stage:`:**
Check the threshold. If exceeded, print the alert, set the `found` flag, and `break` — this exits the loop immediately.

**`if not found:`:**
After the loop, check whether `break` was triggered. If `found` is still False, the stage never reached the threshold during the record period.

**Why `break` instead of a full scan:**
In a real-time monitoring system, we want to issue the alert as early as possible — on the first day the threshold is crossed. Continuing the loop after detecting the event is wasted computation.

### Algorithm

```
1. stage_m = [14 daily values]
   flood_stage = 18.0 m

2. for day, stage in enumerate(stage_m, 1):
     print each day
     if stage >= 18.0:
       print FLOOD ALERT
       found = True
       break

3. After loop:
   if not found: print "No flood stage reached"
```

### Expected output

```
Day 1: 12.4 m — Normal
Day 2: 12.8 m — Normal
...
Day 9: 18.9 m — *** FLOOD STAGE EXCEEDED ***
  First flood stage: Day 9, Stage=18.9 m
  ALERT issued — downstream evacuation protocol activated
```

In [ ]:
stage_m = [
    12.4, 12.8, 13.1, 13.9, 14.6, 15.2, 16.1, 17.3,
    18.9, 20.4, 19.8, 18.3, 17.1, 16.2, 15.4, 14.8,
]

flood_stage    = 18.0
warning_stage  = 15.0
found          = False

for day, stage in enumerate(stage_m, 1):
    if stage < warning_stage:
        note = "Normal"
    elif stage < flood_stage:
        note = "Warning"
    else:
        note = "*** FLOOD STAGE EXCEEDED ***"

    print(f"Day {day:>2}: {stage:.1f} m — {note}")

    if stage >= flood_stage:
        print(f"  First flood stage: Day {day}, Stage={stage:.1f} m")
        print("  ALERT issued — downstream evacuation protocol activated")
        found = True
        break

if not found:
    print("
No flood stage reached during this record period.")

### 🔁 Try this

What if the entire stage record is below flood_stage? Change `flood_stage = 25.0` and re-run.

Does the "No flood stage reached" message appear? This is the `if not found` check working correctly.

---
## Code Block 3 — continue + break: Sensor Data QC

### What this code does

We combine `continue` (skip bad records) and `break` (stop at extreme spike) in a compound QC workflow — the kind of real-time check used in automated water quality monitoring.

### Why each step is taken

**Processing order matters:**
1. Check for -999 (missing) → `continue`
2. Check for negative values (malfunction) → `continue`
3. Compute statistics and check for extreme spike → `break`

Each bad condition is handled before the next check. This avoids computing on corrupted values.

**`break` on extreme spike:**
A spike > 2000 m³/s suggests a sensor malfunction or catastrophic event. In a real monitoring system, processing would stop and an alert would be sent for human verification.

### Expected output

```
  Record  3: MISSING (-999) — skipped
  Record  5: NEGATIVE (-5.2) — skipped
  Record  8: MISSING (-999) — skipped
  Record 10: NEGATIVE (-12.3) — skipped
  Record 11: EXTREME SPIKE (2134.6) — STOP
Valid records: 7   Mean: 583.6   Peak: 1245.6
```

In [ ]:
sensor_readings = [
    245.6, 312.4, -999, 567.8, -5.2, 890.3, 1245.6,
    -999, 1890.4, -12.3, 2134.6, 456.7, 234.5
]

valid_total = 0.0; valid_count = 0; peak = 0.0

for i, Q in enumerate(sensor_readings, 1):
    if Q == -999:
        print(f"  Record {i:>2}: MISSING (-999) — skipped")
        continue

    if Q < 0:
        print(f"  Record {i:>2}: NEGATIVE ({Q}) — skipped")
        continue

    if Q > 2000:
        print(f"  Record {i:>2}: EXTREME SPIKE ({Q}) — STOP")
        break

    valid_total += Q; valid_count += 1
    if Q > peak: peak = Q

mean_val = valid_total / valid_count if valid_count > 0 else 0
print(f"Valid records: {valid_count}   Mean: {mean_val:.1f}   Peak: {peak}")

### 🔁 Try this

Reorder the sensor_readings so that the extreme spike (2134.6) appears at position 3 instead of 11.

- How many valid records are processed now?
- What does this tell you about the importance of real-time QC?

---
## Session Summary — break, continue, pass

| Statement | Where used | Effect |
|---|---|---|
| `continue` | Inside for/while | Skip rest of this iteration |
| `break` | Inside for/while | Exit loop immediately |
| `pass` | Any block | Do nothing (placeholder) |
| `if not found:` after loop | After for/while | Check if break was triggered |

---
## Day 16 Assignment

Water quality data with -999 (missing), negative pH (error), and turbidity > 20 (stop).

Process valid records, compute mean pH and turbidity.

### ▶ Assignment cell

In [ ]:
data = [
    (1, 7.2, 2.3, "GOOD"),   (2, 6.8, -999, "MISSING"),
    (3, 9.2, 4.1, "GOOD"),   (4, 7.5, 12.8, "GOOD"),
    (5, -5, 3.2, "ERROR"),   (6, 7.8, 1.9, "GOOD"),
    (7, 7.1, 8.4, "GOOD"),   (8, 6.5, 22.1, "SUSPECT"),
    (9, 7.4, 3.6, "GOOD"),   (10, 8.1, 5.2, "GOOD"),
]

ph_total=0; turb_total=0; valid=0

for day, pH, turb, flag in data:
    if flag in ("MISSING", "ERROR"):
        print(f"Day {day}: flag={flag} — skipped")
        continue

    if turb > 20:
        print(f"Day {day}: Turbidity {turb} NTU — stopping scan")
        break

    ph_total += pH; turb_total += turb; valid += 1

print(f"Valid: {valid}  Mean pH: {ph_total/valid:.2f}  Mean Turb: {turb_total/valid:.1f} NTU")

---
- [ ] Run all cells — verify outputs match expected outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit2_LoopsDecisions/CE541E08_U2_Day16.ipynb`
- [ ] Commit: `Day 16 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*